# Recursive eraser!

This application recusively deletes generated pages below the given root

Think: `rm -rf pageroot`

In [ ]:
import markupsafe
header=markupsafe.Markup('<span style="color:red;"><b>Unreliable information. For testing purposes only!</b></span><br/>')

In [ ]:
# Pause between Confluence publish operations 
slowdown = 0.25 #0.25

## Load configuration
Be aware not to commit your credentials!

In [ ]:
import yaml
import copy
import logging
log = logging.getLogger(__name__)

with open('geberit.yaml') as f:
    config = yaml.safe_load(f)

conf_conf = config['confluence']
assert conf_conf
assert len(conf_conf['apiurl']) > 0
space_key = conf_conf['space']
root_page = conf_conf['rootpage']

conf_confidential = copy.deepcopy(config)
conf_confidential['confluence']['password'] = '***'
conf_confidential

In [ ]:
# enable overwrite by test infrastructure
confluence_username = config['confluence']['username']
confluence_password = config['confluence']['password']

The [Confluence API](https://github.com/atlassian-api/atlassian-python-api) is embedded as a **git submodule** in the 'lib' folder next to this notebook.

Use `git submodule update --init` to fetch all submodules after a checkout without `--recursive` option.

If the next cell fails, install confluence-api submodule from the repository root with:
`git submodule add -f https://github.com/atlassian-api/atlassian-python-api.git notebooks/contentfactory/lib/atlassian-python-api`

In [ ]:
import sys
import os

library = 'lib/atlassian-python-api'
sys.path.insert(0, os.path.abspath(library))
from atlassian import Confluence

In [ ]:
confluence = Confluence(url=config['confluence']['apiurl'], username=confluence_username, password=confluence_password, cloud=True)
root_page_id = confluence.get_page_id(space_key, config['confluence']['rootpage'])
root_page_id

In [ ]:
from tqdm.notebook import tqdm_notebook
children = confluence.get_page_child_by_type(root_page_id, limit=100000)
with tqdm_notebook(total=len(children), dynamic_ncols=True, unit='Page') as pbar:    
    while True:
        children = confluence.get_page_child_by_type(root_page_id, limit=250)
        if len(children) == 0:
            break
        for child in children:
            pbar.set_description('Removing page {}'.format(child['title']))
            confluence.remove_page(child['id'], recursive=True)
            pbar.update(1)